<img src="https://github.com/nicholasmetherall/digital-earth-pacific-macblue-activities/blob/main/attachments/images/DE_Pacific_banner.JPG?raw=true" width="900"/>

Figure 1.1.a. Jupyter environment + Python notebooks

# Digital Earth Pacific Notebook 1 prepare postcard and load data to csv

The objective of this notebook is to prepare a geomad postcard for your AOI (masking, scaling and loading additional band ratios and spectral indices) and sampling all the datasets into a csv based on your training data geodataframe.

## Step 1.1: Configure the environment

In [1]:
from datetime import datetime
import geopandas as gpd
import numpy as np
import pandas as pd
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import make_classification

In [3]:
# Predefined variable for title and version

# Enter your initials
initials = "mw"

# site
site = "samoa"

# Date
date = datetime.now()

# Make a clean version string
version = f"{initials}-{site}-{date.strftime('%d%m%Y')}"
print(version)

mw-samoa-27022026


### Postcard csv

The objective of this notebook was to train the machine learning model that will allow us to classify an area with land cover classes defined through the training data.

Step 1.2. Input the training data to sample geomad data from the postcard

In [4]:
# joined_df = gpd.read_file(f"training-data/{version}-training.csv")
# joined_df = joined_df.astype("float32")
# joined_df
joined_df = gpd.read_file("training-data/mw-samoa-27022026_postcard_4-training.csv")

In [5]:
joined_df

,lulc_code,nir,red,blue,green,emad,smad,bcmad,nir08,nir09,...,swir22_swir16,mci,ndci,nbi,ndmi,bsi,awei,tc_wetness,y,x
0,5,0.5859,0.5971000000000001,0.5806,0.6032000000000001,0.08767529,1.5938282e-08,2.0028785e-06,0.6082000000000001,0.5093,...,0.9144333225702291,0.9250078939059045,0.029500203169443313,0.02779391022981837,-0.02779391022981837,-0.0012875536480687184,0.42225000000000046,0.1741199700000001,-1543795.0,4251725.0
1,5,0.4178,0.4419,0.4672,0.45570000000000005,0.22546858,1.4609098e-07,7.927856e-06,0.4066,0.3904,...,0.9997855917667239,1.1845761270201305,-0.11225773974326705,0.05496494005881025,-0.05496494005881025,0.012937764889582929,0.396725,0.036713120000000016,-1544315.0,4250775.0
2,5,0.3451,0.1273,0.13540000000000002,0.10210000000000001,0.06835226,1.778841e-07,3.3188785e-06,0.3477,0.2732,...,2.758620689655172,1.0784375,0.4308070646098815,-0.34463276836158185,0.34463276836158185,0.1033774958014555,-0.2633000000000001,-0.09050719000000004,-1544175.0,4250865.0
3,5,0.2732,0.11850000000000001,0.17250000000000001,0.1348,0.08236941,1.7453432e-07,4.613605e-06,0.2939,0.2795,...,1.1458235016517226,1.7335025380710658,0.14161535675479903,-0.1263656977942692,0.1263656977942692,-0.10458488228004954,-0.15744999999999987,-0.06452399000000002,-1544145.0,4250025.0
4,5,0.3521,0.385,0.40850000000000003,0.4062,0.35088086,9.2592835e-08,1.1055483e-05,0.42860000000000004,0.38970000000000005,...,1.1418963254593175,0.720040899795501,0.11899313501144168,0.26775501715711764,-0.26775501715711764,0.17402399956561876,0.15547500000000025,-0.09727388999999989,-1544225.0,4251645.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
373,3,0.35450000000000004,0.1075,0.08080000000000001,0.12350000000000001,0.11958877,4.2413174e-07,5.9985955e-06,0.3688,0.41190000000000004,...,0.6815742397137745,1.886641830761043,0.27217332430602575,-0.11829652996845426,0.11829652996845426,-0.18723578344470207,-0.5138250000000001,-0.07710438000000003,-1546415.0,4218565.0
374,3,0.3705,0.0813,0.0629,0.1068,0.12087271,4.0342212e-07,6.1576393e-06,0.3992,0.2651,...,0.6110476190476191,2.025697102241662,0.38455715367146104,-0.17061611374407581,0.17061611374407581,-0.28395793215819876,-0.5794999999999999,-0.07214240999999996,-1546445.0,4218585.0
375,3,0.40240000000000004,0.0946,0.0801,0.12710000000000002,0.120184734,3.3916533e-07,5.865344e-06,0.4485,0.2922,...,0.5726308086351994,2.3810650887573965,0.2822458270106222,-0.1910611217996153,0.1910611217996153,-0.31543075245365326,-0.576575,-0.07299503000000002,-1545755.0,4218095.0
376,3,0.39830000000000004,0.092,0.07680000000000001,0.12610000000000002,0.17843759,4.4244527e-07,7.716318e-06,0.44420000000000004,0.3829,...,0.6702253855278767,1.7860986547085203,0.4158730158730159,-0.08307273963290285,0.08307273963290285,-0.19808346992813017,-0.6547000000000001,-0.10765574,-1545775.0,4218135.0


In [6]:
joined_df=joined_df.drop(columns=["y", "x"])

In [7]:
print(len(joined_df.columns))
joined_df.columns

33


Index(['lulc_code', 'nir', 'red', 'blue', 'green', 'emad', 'smad', 'bcmad',
       'nir08', 'nir09', 'swir16', 'swir22', 'coastal', 'rededge1', 'rededge2',
       'rededge3', 'mndwi', 'ndti', 'cai', 'ndvi', 'evi', 'savi', 'ndwi',
       'b_g', 'b_r', 'swir22_swir16', 'mci', 'ndci', 'nbi', 'ndmi', 'bsi',
       'awei', 'tc_wetness'],
      dtype='object')

In [8]:
from sklearn.model_selection import train_test_split

training_data, test_data = train_test_split(joined_df, test_size=0.2, random_state=1337)

# The classes are the first column
classes = np.array(training_data)[:, 0]

# The observation data is everything after the second column
observations = np.array(training_data)[:, 1:]

# Create a model...
classifier = RandomForestClassifier(max_depth=4)

# ...and fit it to the data
model = classifier.fit(observations, classes)

In [9]:
# Dynamically create the filename with f-string
file_path = f"models/{version}-test.model"

# Save the model
joblib.dump(model, file_path)

['models/mw-samoa-27022026-test.model']

In [10]:
import pandas as pd
test_actual = np.array(test_data)[:, 0]

test_predicted = model.predict(np.array(test_data)[:,1:])


unique_labels = sorted(np.unique(np.concatenate([np.asarray(test_actual), np.asarray(test_predicted)])))


pd.crosstab(test_actual, test_predicted, margins=True)

col_0,1,2,3,4,5,6,All
row_0,,,,,,,
1,11,1,0,0,0,0,12
2,1,8,3,0,0,0,12
3,0,0,12,0,0,0,12
4,0,0,0,10,0,0,10
5,0,2,0,0,15,0,17
6,0,2,0,0,3,8,13
All,12,13,15,10,18,8,76


In [11]:
from sklearn.metrics import accuracy_score

accuracy_score(test_actual, test_predicted)

0.8421052631578947